# Mastering JSON in Python for AI Engineering

## Learning Goal
- **What this notebook teaches**: How to read, write, and manipulate JSON with Python's built-in `json` module, including handling non-serializable types common in AI work.
- **Background & links**: JSON is defined by [RFC 8259](https://datatracker.ietf.org/doc/html/rfc8259). Python docs: https://docs.python.org/3/library/json.html
- **Why it matters in AI engineering**: Every LLM API (Anthropic, OpenAI, HuggingFace), config file, evaluation log, RAG pipeline, and vector DB schema uses JSON. You will parse and write it hundreds of times per day.

## Mental Model

```
[File / Network]  ──json.load / json.loads──▶  [Python Object]  ──json.dump / json.dumps──▶  [File / Network]
  "text string"                                   dict / list                                    "text string"
```

**JSON is not a data structure — it is a serialization format.**
It is plain text on a wire or disk. You cannot do `response["key"]` on raw JSON.
You must first **parse** it into a Python dict/list using `json.loads()`.

In production: API gives you a string → parse → work with Python dict → serialize back to JSON to store or forward.

## Background Explanation

### JSON's 6 data types

| JSON type | Python equivalent | Example |
|-----------|-----------------|---------|
| string | `str` | `"hello"` |
| number | `int` / `float` | `42`, `3.14` |
| boolean | `bool` | `true` → `True` |
| null | `None` | `null` → `None` |
| array | `list` | `[1, 2, 3]` |
| object | `dict` | `{"key": "value"}` |

### Core functions

| Function | Direction | Typical use |
|----------|-----------|-------------|
| `json.loads(s)` | string → Python | Parse an API response |
| `json.dumps(obj)` | Python → string | Serialize to send/store |
| `json.load(f)` | file → Python | Read a config file |
| `json.dump(obj, f)` | Python → file | Write a config file |

### Key parameters
- `indent=2` — pretty-print with 2-space indent
- `ensure_ascii=False` — allow Arabic / non-ASCII text (critical for Arabic NLP!)
- `default=fn` — custom serializer for non-JSON types (NumPy arrays, datetimes)
- `sort_keys=True` — deterministic output, useful for diffs and caching

In [1]:
import json
print("json module is built-in — no install needed")

json module is built-in — no install needed


## Minimal Working Example

In [4]:
# Simulated API response string (what you receive from requests.get().text)
api_response_str = '''
{
    "model": "claude-sonnet-4-5",
    "usage": {"input_tokens": 512, "output_tokens": 128},
    "content": [{"type": "text", "text": "Hello, Abdulaziz!"}],
    "stop_reason": "end_turn"
}
'''

# Step 1: Parse string → Python dict
response = json.loads(api_response_str)
print(type(response))                          # <class 'dict'>

# Step 2: Access nested fields naturally
tokens_in = response["usage"]["input_tokens"]
text_out  = response["content"][0]["text"]
print(f"Input tokens : {tokens_in}")
print(f"Model output : {text_out}")

# Step 3: Serialize back to string
compact  = json.dumps(response)
readable = json.dumps(response, indent=2)
print(f"\nCompact length : {len(compact)} chars")
print(f"readable length : {len(readable)} chars")
print("Readable preview:")
print(readable[:180], "...")

<class 'dict'>
Input tokens : 512
Model output : Hello, Abdulaziz!

Compact length : 171 chars
readable length : 219 chars
Readable preview:
{
  "model": "claude-sonnet-4-5",
  "usage": {
    "input_tokens": 512,
    "output_tokens": 128
  },
  "content": [
    {
      "type": "text",
      "text": "Hello, Abdulaziz!"
  ...


## Exercise 1 — Parse a Batch of Evaluation Results

**Goal**: Parse a JSON log of LLM evaluation runs and compute summary statistics.

**Real context**: In AI evaluation pipelines you log every LLM call to a JSONL file and later parse it to compute costs, latencies, and pass rates.

In [5]:
# Data setup — run this cell first
eval_log = [
    {"id": 1, "model": "claude-sonnet-4-5", "input_tokens": 320, "output_tokens":  85, "passed": True,  "latency_ms": 420},
    {"id": 2, "model": "claude-sonnet-4-5", "input_tokens": 510, "output_tokens": 140, "passed": False, "latency_ms": 810},
    {"id": 3, "model": "gpt-4o",            "input_tokens": 280, "output_tokens":  60, "passed": True,  "latency_ms": 350},
    {"id": 4, "model": "gpt-4o",            "input_tokens": 640, "output_tokens": 190, "passed": True,  "latency_ms": 920},
    {"id": 5, "model": "claude-sonnet-4-5", "input_tokens": 450, "output_tokens": 110, "passed": True,  "latency_ms": 530},
]
json_log_str = json.dumps(eval_log, indent=2)
print(json_log_str[:300], "...")

[
  {
    "id": 1,
    "model": "claude-sonnet-4-5",
    "input_tokens": 320,
    "output_tokens": 85,
    "passed": true,
    "latency_ms": 420
  },
  {
    "id": 2,
    "model": "claude-sonnet-4-5",
    "input_tokens": 510,
    "output_tokens": 140,
    "passed": false,
    "latency_ms": 810
  },
 ...


In [ ]:
from collections import defaultdict

def analyze_eval_log(json_str: str) -> dict:
    """
    Parse a JSON evaluation log and return a summary dict with:
      - total_runs    : int
      - pass_rate     : float  (0.0 – 1.0, rounded to 2 dp)
      - avg_latency_ms: float  (rounded to 2 dp)
      - total_tokens  : int    (input + output combined)
      - by_model      : dict   mapping model_name -> its pass_rate
    """
    # TODO 1: Parse json_str → Python list
    records = json.loads(json_str)
    # print(len(records))

    # TODO 2: total_runs
    total_runs = len(records)

    # TODO 3: pass_rate
    passed = 0
    for log in records:
        if log['passed']:
          passed += 1
      
    shorter = sum(1 for log in records if log['passed'])
              #  [1 for log in records if log['passed']]
    # Catcher | bucket
        
    pass_rate = sum(1 for log in records if log['passed'])/total_runs

    # TODO 4: avg_latency_ms (round to 2 dp)
    avg_latency_ms = sum(log['latency_ms'] for log in records)/total_runs

    # TODO 5: total_tokens
    total_tokens = sum(log.get('input_tokens') + log.get('output_tokens') for log in records)

    # TODO 6: by_model — group records by model name, compute each model's pass rate
    by_model = None #[temp_list[log.get('model')] = 1 if log.get('passed') for log in records]
    
    group  = defaultdict(list)

    for log in records:
       group[log.get('model')].append(log.get('passed'))

    


    for p,c in group.items():
       print (p)
       print(c)
       print(sum(c))
       print("---")

    


    return {"total_runs": total_runs, "pass_rate": pass_rate,
            "avg_latency_ms": avg_latency_ms, "total_tokens": total_tokens,
            "by_model": by_model}

result = analyze_eval_log(json_log_str)
print(json.dumps(result, indent=2))

claude-sonnet-4-5
[True, False, True]
2
---
gpt-4o
[True, True]
2
---
{
  "total_runs": 5,
  "pass_rate": 0.8,
  "avg_latency_ms": 606.0,
  "total_tokens": 2785,
  "by_model": null
}


In [ ]:
temp_list = {}
temp_list["model"] = 1
# temp_list["model"] = temp_list["model"] + 1
# print(temp_list.get('models'))

temp_list

None


{'model': 1}

**Questions to think about**
1. What happens if `latency_ms` is `null` for some records? How do you guard against it?
2. In production the log would be JSONL (one JSON object per line). How would you modify the parser?
3. Why is `ensure_ascii=False` important when saving Arabic model outputs?

## Solution 1

In [54]:
from collections import defaultdict

def analyze_eval_log(json_str: str) -> dict:
    records        = json.loads(json_str)                                     # 1
    total_runs     = len(records)                                             # 2
    pass_rate      = round(sum(r["passed"] for r in records) / total_runs, 2) # 3
    avg_latency_ms = round(sum(r["latency_ms"] for r in records) / total_runs, 2) # 4
    total_tokens   = sum(r["input_tokens"] + r["output_tokens"] for r in records) # 5

    groups = defaultdict(list)                                                # 6
    for r in records:
        groups[r["model"]].append(r["passed"])
    by_model = {m: round(sum(p)/len(p), 2) for m, p in groups.items()}

    return {"total_runs": total_runs, "pass_rate": pass_rate,
            "avg_latency_ms": avg_latency_ms, "total_tokens": total_tokens,
            "by_model": by_model}

print(json.dumps(analyze_eval_log(json_log_str), indent=2))

{
  "total_runs": 5,
  "pass_rate": 0.8,
  "avg_latency_ms": 606.0,
  "total_tokens": 2785,
  "by_model": {
    "claude-sonnet-4-5": 0.67,
    "gpt-4o": 1.0
  }
}


## Exercise 2 — Custom JSON Serializer for AI Objects

**Goal**: In real AI systems you need to serialize objects JSON doesn't natively support: NumPy arrays, datetimes, dataclasses.

**Why it matters**: Embedding vectors, timestamps, and model configs must be logged. The default serializer raises `TypeError`. You need a `default=` handler.

In [ ]:
import numpy as np
from datetime import datetime

experiment_result = {
    "run_id"   : "exp_arabic_001",
    "timestamp": datetime.now(),                      # not JSON-serializable!
    "embedding": np.array([0.12, -0.85, 0.33, 0.71]),# not JSON-serializable!
    "score"    : float(np.float32(0.9231)),           # np.float32 also fails!
    "config"   : {"model": "arabic-bert", "dim": 768},
}

try:
    json.dumps(experiment_result)
except TypeError as e:
    print(f"TypeError raised: {e}")

In [ ]:
def ai_json_encoder(obj):
    """
    Custom default= function for json.dumps.
    Handle: np.ndarray, numpy scalar types, datetime.
    """
    # TODO 1: numpy arrays → Python list
    # Hint: isinstance(obj, np.ndarray)

    # TODO 2: numpy scalars (np.float32, np.int64, etc.)
    # Hint: np.floating and np.integer are base classes

    # TODO 3: datetime → ISO 8601 string
    # Hint: obj.isoformat()

    raise TypeError(f"Type {type(obj)} not serializable")

output = json.dumps(experiment_result, default=ai_json_encoder, indent=2)
print(output)

## Solution 2

In [ ]:
def ai_json_encoder(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()            # 1 — array → Python list
    if isinstance(obj, np.floating):
        return float(obj)             # 2a — np float → Python float
    if isinstance(obj, np.integer):
        return int(obj)               # 2b — np int → Python int
    if isinstance(obj, datetime):
        return obj.isoformat()        # 3 — datetime → ISO string
    raise TypeError(f"Type {type(obj)} not serializable")

output = json.dumps(experiment_result, default=ai_json_encoder, indent=2)
print(output)

# Verify round-trip (embedding becomes list — expected)
parsed = json.loads(output)
print("\nEmbedding type after round-trip:", type(parsed["embedding"]))

## Common Mistakes

### Mistake 1 — Indexing a JSON string directly (most common beginner error)

In [ ]:
raw = '{"key": "value"}'

# WRONG
try:
    print(raw["key"])
except TypeError as e:
    print(f"ERROR: {e}")

# CORRECT
parsed = json.loads(raw)
print("OK:", parsed["key"])

### Mistake 2 — Mixing `json.load` and `json.loads`

In [ ]:
import io

json.loads('{"a": 1}')         # loads = expects a STRING  (s = string)
json.load(io.StringIO('{"a": 1}'))  # load  = expects a FILE OBJECT

try:
    json.load('{"a": 1}')      # passing a string to load() → AttributeError
except AttributeError as e:
    print(f"AttributeError: {e}")

### Mistake 3 — Arabic text mangled by `ensure_ascii=True` (default)

In [ ]:
arabic_data = {"model": "AraGPT2", "output": "مرحباً بك في عالم الذكاء الاصطناعي"}

print("Default (ensure_ascii=True) — garbled:")
print(json.dumps(arabic_data))

print("\nWith ensure_ascii=False — correct:")
print(json.dumps(arabic_data, ensure_ascii=False, indent=2))

## Real AI Engineering Usage

```python
# RAG — store document chunk with embedding
chunk = {"text": "...", "source": "doc.pdf", "page": 3, "embedding": [...]}
json.dump(chunk, f, ensure_ascii=False)   # Arabic sources need ensure_ascii=False

# LLM API call — every call is JSON over HTTP
payload  = {"model": "claude-sonnet-4-5", "messages": [...], "max_tokens": 1024}
response = requests.post(url, json=payload)  # requests calls json.dumps internally
data     = response.json()                   # requests calls json.loads internally

# Evaluation logging — JSONL format (one JSON per line)
with open("eval.jsonl", "a") as f:
    f.write(json.dumps(result, default=ai_json_encoder) + "\n")

# Config files — load training hyperparameters
with open("config.json") as f:
    config = json.load(f)
lr = config["training"]["learning_rate"]

# Vector DB metadata — Pinecone / Qdrant / Weaviate all use JSON
metadata = {"language": "ar", "domain": "news", "date": "2025-05-17"}
index.upsert(vectors=[(doc_id, embedding, metadata)])
```

## Final Mini Exercise — Production JSONL Report Generator

Build a function that reads a JSONL evaluation log (one JSON per line) and returns a formatted Markdown report string showing: total runs, overall pass rate, per-model table, and a list of slow runs (`latency_ms > 800`).

In [ ]:
jsonl_log = """{"id":1,"model":"claude-sonnet-4-5","passed":true,"latency_ms":420,"tokens":405}
{"id":2,"model":"claude-sonnet-4-5","passed":false,"latency_ms":810,"tokens":650}
{"id":3,"model":"gpt-4o","passed":true,"latency_ms":350,"tokens":340}
{"id":4,"model":"gpt-4o","passed":true,"latency_ms":920,"tokens":830}
{"id":5,"model":"arabic-llm","passed":true,"latency_ms":610,"tokens":560}"""

def jsonl_report(log: str) -> str:
    # TODO: parse each line, compute stats, return a Markdown string
    pass

print(jsonl_report(jsonl_log))

## Key Takeaways

- **JSON is text**, not a data structure. Always parse with `json.loads()` before accessing fields.
- **4 functions**: `loads` / `dumps` for strings, `load` / `dump` for files. The `s` = string.
- Always use **`ensure_ascii=False`** with Arabic, Chinese, or any non-Latin text.
- Use **`default=`** to handle NumPy arrays, datetimes, and custom objects.
- In production: every API response, config, and evaluation log is JSON — master it completely.
- **JSONL** (one JSON per line) is the standard format for large-scale logging and dataset storage.
- JSON does **not** preserve `tuple`, `set`, or `np.ndarray` — they all become `list` after round-trip.